# Image Processing in Python
## Tutorial for TSBB15


## Getting Started
In python we need to import packages which provide the functionality we want.
In this course we will use the numpy, scipy, and matplotlib packages a lot, lets import them!

In [ ]:
import numpy as np
import scipy
from matplotlib import pyplot as plt
plt.rcParams["figure.figsize"] = (10,10)

Numpy lets us work with (num)bers in (py)thon!

It's great for working with vectors and matrices, lets see how to create one:

In [ ]:
row_vector = np.array([[1 ,2 ,3]],dtype=np.float)

Everything in numpy is an array, and arrays have:

**datatype**: e.g, float, int, bool - Describes what types of numbers are in the array

**shape/size**: describes how the array "looks" 

In [ ]:
row_vector.dtype

In [ ]:
row_vector.shape

## Operations
We can **add** and **subtract** easily:

In [ ]:
row_vector+row_vector

In [ ]:
row_vector-row_vector

What about **elementwise multiplication** and **matrix multiplication**?

In [ ]:
row_vector*row_vector #elementwise multiplication, same as .* in matlab!

In [ ]:
row_vector@(row_vector.T) # matrix multiplication, same as * in matlab!

**Task 1**: What does .T do to the array?

**Answer**: Transponation

We can basically do any operation elementwise, by calling the proper numpy function:

In [ ]:
x = np.linspace(0,2*np.pi,num=100)
y = np.sin(x)
plt.plot(x,y) # Here we use matplotlib, which was imported above
plt.show()

### Indexing

In the simplest case we want a single index:

In [ ]:
ind = 0 # Arrays start at 0 (!)
x[ind]

We may also the first 10 elements

In [ ]:
x[0:10]

This is called "slicing".

We can also want every second index:

In [ ]:
x[::2] # short for x[0:-1:2], where -1 is short for the last element of x

**Task 2**: Now we want to get every third index of x, starting at index 5, give the appropriate command below.

In [ ]:
#answer here

### Boolean Indexing

Lets say we want to take all the values of x, where $y<0$.

Then we can use boolean indexing, and it's super simple!

In [ ]:
x[y<0]

In [ ]:
plt.scatter(x[y<0],y[y<0])
plt.show()

**Task 3**: Now we want to index all values of x,y such that $y^2 < 0.5$. Provide the appropriate code below.

In [ ]:
#answer here

### Multi-dimensional indexing

Indexing works basically the same!

In [ ]:
x = np.array([[1,2,3],[4,5,6],[7,8,9]])
x[0:2,0:2] # Selects a 2x2 block.

### Images

Luckily for us, images are arrays so everything written above applies for images too!

We begin by showing some ways of loading images into python.

We start with using **matplotlib** to load the image.

In [ ]:
image_mpl = plt.imread('/courses/TSBB15/images/capybaras.jpg')

plt.imshow(image_mpl) #shows some cute capybaras
plt.show()

Another way is with **opencv**.

The image is automatically made into a numpy array, however with BGR instead of RGB color channels.

In [ ]:
import cv2
image_cv2 = cv2.imread('/courses/TSBB15/images/capybaras.jpg')
plt.imshow(image_cv2) #Now the capybaras are blue, since opencv uses bgr instead of rgb
plt.show()

We can also use **PIL**

The image is not automatically made into a numpy array, but we can convert it quite easily :)

In [ ]:
from PIL import Image
image_pil = Image.open('/courses/TSBB15/images/capybaras.jpg')
print(image_pil)
image_pil = np.array(image_pil)
plt.imshow(image_pil) #Now the capybaras are blue, since opencv uses bgr instead of rgb
plt.show()

**Task 4**: What is the datatype of the images, and what is the shape? What are the maximum and minimum values?

**Answer**:

## Making a 2d filter

Often we want to filter our images, lets make a gaussian blur filter with numpy!

In [ ]:
kernel_size = 13;
(x,y) = np.meshgrid(np.linspace(-1,1,num=kernel_size),np.linspace(-1,1,num=kernel_size))

We just created a meshgrid! You can think of this as an array containing all the coordinates.

In [ ]:
plt.imshow(x)
plt.colorbar()
plt.show()

In [ ]:
plt.imshow(y)#Note that the y-axis goes down!
plt.colorbar()
plt.show()

Now we want to create a gaussian on this grid!

**Task 5**: The equation for a gaussian filter is $\exp(\frac{-(x^2+y^2)}{2\sigma^2})$ 

Lets assume that $\sigma=1$. How can you write this in numpy? Write the code below. 

*Hint*: $\exp$ can be written np.exp() in numpy and $x^2$ can be written x**2

In [ ]:
lp_filter = #answer here

Additionally, we want the filter to sum to 1, to ensure that we don't make the image super bright or dark!

In [ ]:
lp_filter = lp_filter/np.sum(lp_filter)

In [ ]:
plt.imshow(lp_filter)
plt.show()

## Convolving our filter with the image

Now that we (hopefully) have a working low-pass filter. 

Lets convolve it with the capybara image!

First lets import a package for performing convolution

In [ ]:
from scipy.signal import convolve2d as conv2

Now lets try convolving!

In [ ]:
image_lp = conv2(image_pil,lp_filter, mode='same') # uhoh...

Our capybaras are in color! This means that they are technically 3D arrays. 

An easy way to fix this issue is to do the convolution seperately for each color channel. 

**Task 6**: Convolve the capybary image seperately for each color channel below.

In [ ]:
image_lp_r = conv2(image_pil[:,:,0],lp_filter, mode='same')#here's how to do it for the red channel
image_lp_g = #do it for the green channel!
image_lp_b = #and for the blue

When we have multiple numpy arrays we can *stack* them together, to get a combined array.

Note: Make sure to remember which dimension you stack them in! This can be selected by the *axis* parameter in the np.stack function 

In [ ]:
image_lp = np.stack((image_lp_r,image_lp_g,image_lp_b),axis=-1)
image_lp.shape

So lets show our beautiful resulting image!

In [ ]:
plt.imshow(image_lp) # uhoh...
plt.show()

Sadly it was not to be.

You may recall that the image had values between [0,255] and was of type uint8.

When we convolved it the datatype became float, and now matplotlib is telling us that our image is weird.

In [ ]:
plt.imshow(image_lp/255.0) # The easiest way is to simply divide the image by 255
plt.show()

In [ ]:
plt.imshow(image_lp.astype(np.uint8)) # We can also convert it back to uint8!
plt.show()

## Fourier Domain

You may have noticed that the convolution was a bit slow.

Since the image is very large, and the kernel is also quite large, it is quite computationally expensive.

A faster way could be to do it in the fourier domain!

Remember that $f*g = \mathcal{F}^{-1}[FG]$

In [ ]:
# Take the fourier transfrom of the image
f_image_pil_r = np.fft.fft2(image_pil[:,:,0])
f_image_pil_g = np.fft.fft2(image_pil[:,:,1])
f_image_pil_b = np.fft.fft2(image_pil[:,:,2])

In [ ]:
# Fourier transform of filter, note that we MUST have the same shape as the image(!!)
zero_padded_lp_filter = np.zeros(image_pil[:,:,0].shape,dtype=np.float)
zero_padded_lp_filter[:kernel_size,:kernel_size] = lp_filter # here we have implicitly shifted the filter
zero_padded_lp_filter = np.roll(zero_padded_lp_filter,shift=(-(kernel_size//2),-(kernel_size//2)),axis=(0,1))#so we need to roll it back into place!
f_lp_filter = np.fft.fft2(zero_padded_lp_filter)

**Task 7**: Now multiply and do the inverse transform for each channel!
    
*Hint*: the inverse transform is np.fft.ifft2

In [ ]:
image_lp_r_fast = #answer here
image_lp_g_fast = #answer here
image_lp_b_fast = #answer here

**Task 8**: Stack the channels as you did before. 

In [ ]:
image_lp_fast = #answer here

In [ ]:
plt.imshow(image_lp_fast.astype(np.uint8)) # We can also convert it back to uint8!
plt.show()

**Check**: Does the image look basically identical to the previous ones?

That was basically it for the basics, however, there are infinitly many things to know.
Here is a neat cheat-sheet:
http://datacamp-community-prod.s3.amazonaws.com/ba1fe95a-8b70-4d2f-95b0-bc954e9071b0

And here is some more advanced operations that Johan Edstedt put in a github gist:

https://gist.github.com/Parskatt/18a7db9784a43aa598e59f21412b4ec2

Below we will go through some stuff that should help you with Lab 1 and Lab 2!

## Helpful stuff for lab 1 and lab 2 (Dont Miss These)

## Lab 1
In Lab 1 you will construct a regularized derivative filter. 

This filter arises from the fact that:
$\frac{\partial}{\partial x} (f*g(\sigma))(x) = (f*\frac{\partial}{\partial x}g(\sigma))(x) = (f*\frac{-x}{\sigma^2}g(\sigma))(x)$

Where $g=\exp(-\frac{x^2}{2\sigma^2})$. 

So we use the filter: $df = \frac{-x}{\sigma^2}g(\sigma)$ and convolve it with our signal.

Furthermore this filter is *seperable* when its in 2d, which is great!

Because since $df(x,y) = df(x)g(y)$, then $f*df=(f*df(x))*g(y)$

So we can make our 2d convolution (which takes scales quadratically with kernel size) and compose it to 2 seperate convolution which only scales linearly!

In [ ]:
kernel_size = 7
sigma = 1.5
x = np.linspace(-(kernel_size//2),kernel_size//2,num=kernel_size)[np.newaxis,:]
g = np.exp(-x**2/(2*sigma**2))
g = g/np.sum(g)
df = -x*g/(sigma**2)

In [ ]:
plt.plot(x[0],df[0])
plt.show()

It is veeeeery important that our filter is **symmetric**.

**Task 9**: Explain why the filter is not symmetric if the kernel size is even!

*Hint*: Try to change the kernel size above, and see what happens :)

**Answer**:

Below we show you how to perform the convolution and the resulting derivative image

In [ ]:
image_pil = Image.open('/courses/TSBB15/images/capybaras.jpg')
image_pil_gray = np.mean(np.array(image_pil),axis=-1)
dx = conv2(conv2(image_pil_gray,df,mode='same'),g.T,mode='same')
dy = conv2(conv2(image_pil_gray,df.T,mode='same'),g,mode='same')

In [ ]:
plt.figure()
plt.subplot(1,2,1)
plt.imshow(dx)
plt.subplot(1,2,2)
plt.imshow(dy)
plt.show()

## Lab 2

In Lab 2 you will use the gopimage color palette for visualization.

This uses a complex representation of the images.

In [ ]:
z = dx + 1j*dy
from lab2 import gopimage
gopimage(z)#You should see (parts of) a colorful and noisy capybara

While the above is quite pretty, it is quite noisy!

We can instead look at a better representation, e.g. the structure tensor!

In [ ]:
T = np.zeros_like(image_pil)
Tlp = np.zeros_like(T)
T [: ,: ,0] = dx * dx
T [: ,: ,1] = dx * dy
T [: ,: ,2] = dy * dy

In [ ]:
sigma = 3.0
kernel_size = 25
lp = np.exp(-0.5*(np.linspace(-(kernel_size//2) ,kernel_size//2,num=kernel_size)/ sigma )**2)[np.newaxis,:]
lp = lp/np.sum(lp) # normalize the filter
Tlp[: ,: ,0] = conv2 ( conv2 ( T [: ,: ,0] , lp , mode ='same') , lp.T , mode ='same')
Tlp[: ,: ,1] = conv2 ( conv2 ( T [: ,: ,1] , lp , mode ='same') , lp.T , mode ='same')
Tlp[: ,: ,2] = conv2 ( conv2 ( T [: ,: ,2] , lp , mode ='same') , lp.T , mode ='same')

In [ ]:
z = Tlp[:,:,0]-Tlp[:,:,2]+2j*Tlp[:,:,1]

In [ ]:
gopimage(z)#You should see (parts of) a colorful and noisy capybara

Note that the colors here might be slightly different from what is presented in the lecture.

One difference might come from the fact that the y-axis is directed downwards.

**Task 10**: Change the direction of the y-axis (e.g. by flipping the direction of the filter), and plot z again. What happens?

**Answer**:

**Task 11**: What happens when you increase the value of sigma? What happens if you decrease it? Why?

**Answer**: